# Cuál es el problema?

Abandono de clientes (Churn)

Trabajamos con una empresa de telecomunicaciones que quiere ***predecir cuales clientes van a abandonar el servicio*** (churn) para poder tomar acciones preventivas.

Por qué es importante?

* Conseguir un cliente nuevo representa entre 5x y 25x más que retener uno existente.
* Si podemos predecir quien se va, podemos ofrecerle un descuento o llamarlo antes de que cancele.



In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer # Para transformar columnas específicas
from sklearn.preprocessing import OneHotEncoder # Para escalar y codificar variables
from sklearn.linear_model import LogisticRegression # Modelo de regresión logística

# Metricas de evaluación para la clasificación

from sklearn.metrics import (
    confusion_matrix, # Matriz de confusión
    classification_report, # reporte con precisión, recall y f1-score
    accuracy_score, # Porcentaje de predicciones correctas
)

import matplotlib.pyplot as plt
print("Librerias cargadas correctamente")

In [ ]:
%pip install pandas numpy scikit-learn matplotlib

In [ ]:
#Cargar el archivo CSV en un DataFrame (tabla de datos  )
df = pd.read_csv("abandono_clientes.csv")
# Mostrar las primeras filas del DataFrame para verificar que se cargó correctamente
print("primeras 5 filas del DataSet:")
df.head()

In [ ]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

In [ ]:
conteo = df["abandono"].value_counts()
porcentaje = df["abandono"].value_counts(normalize=True) * 100

print("Distribucion de la variable objetivo (abandono):")
print(f"  Permanece (0): {conteo[0]} clientes ({porcentaje[0]:.1f}%)")
print(f"  Abandona  (1): {conteo[1]} clientes ({porcentaje[1]:.1f}%)")
print()
print("Las clases estan moderadamente desbalanceadas.")
print("Hay mas clientes que permanecen que clientes que abandonan.")
print("Esto es lo tipico en la realidad: la mayoria de clientes NO se van.")

# Analisis exporatorio: Como se comportan las variables segun abandono?

## Antes de entrana el modelo, vemos si las variables tiene relación con el abandono. Esto nos ayuda a entender que factores podrian influir en la decisión del cliente

In [ ]:
#Comporar el promedio de cada variable 

comparacion = df.groupby("abandono")[["antiguedad_meses","factura_mensual",
                                      "llamadas_soporte","satisfaccion"]].mean().round(2)
comparacion.index=["Permanece (0)","Abandona (1)"]
print("Promedio de variables numerica segun abandono:")
comparacion

In [ ]:
# Analisi del tipo contrato vs el abandono
tabla_contrato = pd.crosstab(
    df["contrato"], 
    df["abandono"], 
    normalize="index"
    ) * 100
tabla_contrato.columns = ["Permanece (%)", "Abandona (%)"]
print("porcentaje de abandono pro tipo de contrato:")
comparacion 

# Preapara los datos para el modelo.

1. Separar variables predictoras (x) de la variable objetivo (y)
2. Entrenar los datoas con el (80%) del dataset y prueba con el (20%) del dataset



In [ ]:
# Paso 1 separar X(vriables predictoas) e y (variabl objetivo)
X= df[["antiguedad_meses","factura_mensual","llamadas_soporte",
        "satisfaccion","contrato"]]
y=df["abandono"]

# Paso 2 Dividir el conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
     test_size=0.2, # 20% para prueba
     random_state=42, # 80% para reproducibilidad
        stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} Clientes")
print(f"Datos de prueba: {X_test.shape[0]} Clientes")
print("\n")
print(f"Proporcion de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporcion de abandono en prueba: {y_test.mean()*100:.1f}%")
print()
print("La proporciones son similares gracias al parametro 'stratify=y' en train_test_split")

# Construir  el pipeline y entrenar el modelo

1. Calula una puntuación lineal
2.Pasa por una puntación que la convierte en probabilidad entre 0 y 1
3. SI la probabilidad es mayor a 0.5  predice la clase 1 (abandona): si no, predice 0 (permanece)

In [ ]:
from sklearn.pipeline import Pipeline

numeric_features = ["antiguedad_meses", "factura_mensual",
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]


preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)


pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

# Entrenar el modelo con los datos de entrenamiento
pipe.fit(X_train, y_train)

print("Modelo entranado correctamente.")


# Hacer predicciones
Usamos el modulo entranado para predecir el abandono en los datos de pruebas 

In [ ]:
# Generar prediciones con los datos de prueba
y_pred = pipe.predict(X_test)

# Mostrar las primeras 10 predicciones
comparacion_pred = pd.DataFrame({
    "Real": y_test.values[:10],
    "Prediccion": y_pred[:10],
    "Correcto": ["Si" if r == p else "No" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("Primeras 10 predicciones vs realidad:")
print()
comparacion_pred

# Evaluación del modulo:  exactitud/presución (Acurracy)

Cuando hablamos del porcentaje de presición nos referimos a las predicciones que fueron correctas

Accurracy= predicciones Correctas /Total de predicciones.

Ejemplo: Si el 95% de clientes **NO** abandona, un modelo que siempre diga "No abandona" tendria un 95% de presicion, pero jamas detectaria a un cliente en riegos. seria inutil.





In [ ]:
acc=accuracy_score(y_test, y_pred)
print(f"Accuracy (exactitud): {acc:.4f} ({acc*100:.2f}%%)")
print()
print(f"Esto significa que el modulo acerto {acc*100:.2f}% predicciones.")